# Guided exercise: Evaluate retrieval quality

## Objective

Implement a small, deterministic evaluator for one ranked retrieval query. It will calculate **Precision@k**, **Recall@k**, and **Reciprocal Rank@k** from retrieved source IDs and known relevant source IDs.

This is the first feature developed with a reusable guided-TP workflow. Future roadmap items will use the same approach: a scoped exercise, acceptance tests, and documentation before functional implementation. Work through the steps in order. Ask for a hint only when a step is unclear; after each step, pause and review it together before moving on.

## Your role and scope

Implement the function in `src/devassistant/evaluation.py`. The function signature and acceptance tests are already provided. Do not change the tests to make them pass. Keep this exercise focused on **one query**; dataset loading, aggregate reports, charts, and changes to the retrieval algorithm are out of scope.

No LM Studio server, network access, or model inference is needed.

## Project context

Recall ranks Markdown chunks, and several returned chunks may come from the same source document. To evaluate retrieval at the source level, the evaluator receives source IDs (for example, `python.md`) in ranked order. The relevant set is the collection of source documents judged useful for that question.

The current search scores are similarity scores, not calibrated probabilities. These evaluation metrics measure ranking against known relevant sources; they do not prove that generated answers are correct or grounded.

## Function contract

```python
evaluate_retrieval(
    retrieved_ids: list[str],
    relevant_ids: set[str],
    k: int = 5,
) -> dict[str, float]
```

- `retrieved_ids` is already ranked from highest to lowest; use at most its first `k` unique IDs.
- If an ID appears more than once, keep its first occurrence. Repeated chunks from one source must not inflate document-level scores.
- `relevant_ids` contains the source IDs judged relevant for this query and must not be empty.
- `k` must be a positive integer. A boolean is not accepted as an integer cutoff.
- Return exactly these keys: `precision_at_k`, `recall_at_k`, and `reciprocal_rank_at_k`.
- Every metric is between 0 and 1. If retrieval is empty, all three values are 0.
- Precision@k is divided by `k`, even when fewer than `k` unique IDs were retrieved.
- Invalid `k` or an empty `relevant_ids` raises `ValueError`.

## Step 1 — Derive the metrics

For a cutoff `k`, let `hits` be the number of unique relevant IDs among the first `k` unique retrieved IDs. Let `R` be the total number of relevant IDs.

1. **Precision@k** asks: what fraction of the first `k` result slots are relevant? Use `hits / k`.
2. **Recall@k** asks: what fraction of all known relevant sources did we retrieve? Use `hits / R`.
3. **Reciprocal Rank@k** asks how early the first relevant source appears. It is `1 / rank` for the first relevant ID within the cutoff, otherwise `0`. Ranks start at 1.

Work through this example before coding:

- Ranked results: `[noise.md, python.md, rag.md]`
- Relevant sources: `{python.md, rag.md}`
- Cutoff: `k = 2`

Write down `hits`, Precision@k, Recall@k, and Reciprocal Rank@k. The tests include this case so you can check your calculation.

In [2]:
# Step 1 scratchpad: calculate these values by hand before coding.
retrieved_ids = ["noise.md", "python.md", "rag.md"]
relevant_ids = {"python.md", "rag.md"}
k = 2

# Write your answers in the markdown cell above:
hits = len(set(retrieved_ids[:k]) & relevant_ids)
precision_at_k = hits / k if k else 0

R = len(relevant_ids)
recall_at_k = hits / R

rank = 0
for cur_rank, source_id in enumerate(retrieved_ids[:k], start=1):
    if source_id in relevant_ids:
        rank = cur_rank
        break

reciprocal_rank_at_k = 1 / rank if rank else 0

## Step 2 — Plan the algorithm

Before writing code, answer these questions in your own words:

1. How will you remove duplicate IDs while preserving their first rank?
2. Which sequence should be cut to the first `k` entries?
3. How can you count hits without repeatedly scanning the relevant collection?
4. How will you find the rank of the first relevant result?
5. What values should be returned when no results are retrieved?

### Hint 1

A `set` can tell you quickly whether an ID has already been seen, while a list can preserve the original ranking. Do not sort the input: its order is meaningful.

### Hint 2

After removing duplicates, inspect only the first `k` entries. The position of the first relevant entry is already its one-based rank if you start counting at 1.

Suggested decomposition: validate inputs, build the unique ranked list, select the cutoff, count hits, locate the first hit, and calculate the three metrics. You can implement this in one function; no helper abstraction is required.

## Answers

1. Use a `set` to track IDs that have already been seen and a `list` to preserve the original ranking order. Keep only the first occurrence of each ID.

2. Remove duplicate IDs first, then use list slicing (`unique_ids[:k]`) to keep at most the first `k` unique documents.

3. Count the relevant results among the first `k` unique IDs by checking whether each ID belongs to `relevant_ids`. Since `relevant_ids` is a set, membership checks are efficient on average.

4. Iterate through the first `k` unique results in their original order using `enumerate(..., start=1)`. Stop at the first relevant ID and use its one-based position as its rank. No sorting is needed because the results are already ranked.

5. If no results are retrieved, return `0.0` for all three metrics: `precision_at_k`, `recall_at_k`, and `reciprocal_rank_at_k`.

## Step 3 — Implement and test

Complete `evaluate_retrieval()` in `src/devassistant/evaluation.py`. Run the focused tests first:

```powershell
pytest tests/test_retrieval_evaluation.py -q -p no:cacheprovider --tb=short
```

The tests cover a perfect ranking, partial hits, the cutoff, duplicate source IDs, fewer results than `k`, empty results, and invalid inputs. Once they pass, run the whole suite:

```powershell
pytest -q -p no:cacheprovider --basetemp=.pytest-tmp --tb=short
```

In [ ]:
# Step 3: run this cell after implementing evaluate_retrieval().
from devassistant.evaluation import evaluate_retrieval

evaluate_retrieval(
    retrieved_ids=["noise.md", "python.md", "rag.md"],
    relevant_ids={"python.md", "rag.md"},
    k=2,
)

NotImplementedError: Implement the retrieval metrics described in the guided TP.

## Step 4 — Connect the evaluator to Recall conceptually

Do not change the search pipeline in this exercise. Explain how a future evaluation dataset could map each question to relevant source IDs and how `search()` results could be reduced to ranked, unique source IDs. Keep that mapping as a design note; a dataset runner is a later roadmap step.

## Review questions

- Why does Precision@k use `k` as its denominator when fewer than `k` results exist?
- Why must duplicate chunks from one source not count as separate relevant documents?
- What does a low Recall@k tell you that a low Reciprocal Rank@k does not?
- Why do good retrieval metrics still not guarantee a correct generated answer?